# Hybrid Control via Behavior Cloning and Residual RL

This notebook follows the proposal structure: train and freeze BC, wrap the residual MDP, then compare MLP+PPO against action-chunk BC+SAC.

## Phase I: Expert Demonstrations, Single-Step BC, and Action-Chunk BC

In [ ]:
!pip install -q stable-baselines3[extra] gymnasium tensorboard

from google.colab import drive
drive.mount("/content/drive")

import os
import random
from collections import defaultdict

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from torch.utils.data import DataLoader, TensorDataset


SEED = 7
ENV_NAME = "Pendulum-v1"
BC_HIDDEN_DIMS = (256, 256)
ACTION_CHUNK_SIZE = 8

DRIVE_DIR = "/content/drive/MyDrive/BC_RL_project"
os.makedirs(DRIVE_DIR, exist_ok=True)

BC_PATH = os.path.join(DRIVE_DIR, "bc_expert.pth")
ACTION_CHUNK_BC_PATH = os.path.join(DRIVE_DIR, "bc_action_chunk.pth")
DEMO_PATH = os.path.join(DRIVE_DIR, "expert_demonstrations.npz")

TEACHER_TIMESTEPS = 300_000
EXPERT_TRANSITIONS = 30_000
BC_EPOCHS = 100
CHUNK_BC_EPOCHS = 100


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalized_to_physical(normalized_action, action_space):
    normalized_action = np.asarray(normalized_action, dtype=np.float32)
    action_low = action_space.low.astype(np.float32)
    action_high = action_space.high.astype(np.float32)
    midpoint = (action_high + action_low) / 2.0
    scale = (action_high - action_low) / 2.0
    return midpoint + np.clip(normalized_action, -1.0, 1.0) * scale


def physical_to_normalized(physical_action, action_space):
    physical_action = np.asarray(physical_action, dtype=np.float32)
    action_low = action_space.low.astype(np.float32)
    action_high = action_space.high.astype(np.float32)
    midpoint = (action_high + action_low) / 2.0
    scale = (action_high - action_low) / 2.0
    return np.clip((physical_action - midpoint) / (scale + 1e-8), -1.0, 1.0)


class BCPolicy(nn.Module):
    """Single-step behavior cloning policy: s_t -> a_t."""

    def __init__(self, state_dim, action_dim, hidden_dims=BC_HIDDEN_DIMS):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
        )
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dims[1], action_dim),
            nn.Tanh(),
        )

    def forward_tensor(self, state_tensor):
        return self.policy_head(self.encoder(state_tensor))

    def forward(self, state_tensor):
        return self.forward_tensor(state_tensor)

    def predict(self, state):
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32)
            squeeze = state_tensor.ndim == 1
            if squeeze:
                state_tensor = state_tensor.unsqueeze(0)
            action = self.forward_tensor(state_tensor).cpu().numpy().astype(np.float32)
        return action[0] if squeeze else action

    def freeze(self):
        self.eval()
        for param in self.parameters():
            param.requires_grad = False
        return self


class ActionChunkBC(nn.Module):
    """Action-chunking BC policy: s_t -> [a_t, ..., a_{t+k-1}]."""

    def __init__(self, state_dim, action_dim, chunk_size=ACTION_CHUNK_SIZE, hidden_dims=BC_HIDDEN_DIMS):
        super().__init__()
        self.action_dim = action_dim
        self.chunk_size = chunk_size
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
        )
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dims[1], chunk_size * action_dim),
            nn.Tanh(),
        )

    def forward_tensor(self, state_tensor):
        flat = self.policy_head(self.encoder(state_tensor))
        return flat.view(-1, self.chunk_size, self.action_dim)

    def forward(self, state_tensor):
        return self.forward_tensor(state_tensor)

    def predict_chunk(self, state):
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32)
            if state_tensor.ndim == 1:
                state_tensor = state_tensor.unsqueeze(0)
            chunk = self.forward_tensor(state_tensor)[0].cpu().numpy().astype(np.float32)
        return chunk

    def predict(self, state):
        return self.predict_chunk(state)[0]

    def freeze(self):
        self.eval()
        for param in self.parameters():
            param.requires_grad = False
        return self


class ChunkedBCController:
    """Executes a predicted action chunk open-loop, then replans."""

    def __init__(self, chunk_policy):
        self.chunk_policy = chunk_policy
        self.reset()

    def reset(self):
        self._chunk = None
        self._idx = 0

    def predict(self, obs):
        if self._chunk is None or self._idx >= len(self._chunk):
            self._chunk = self.chunk_policy.predict_chunk(obs)
            self._idx = 0
        action = self._chunk[self._idx]
        self._idx += 1
        return action


def train_teacher(total_timesteps=TEACHER_TIMESTEPS):
    env = gym.make(ENV_NAME)
    teacher = PPO("MlpPolicy", env, verbose=0, seed=SEED, device="cpu")
    teacher.learn(total_timesteps=total_timesteps)
    env.close()
    return teacher


def collect_expert_episodes(env_name, teacher, num_transitions=EXPERT_TRANSITIONS):
    env = gym.make(env_name)
    episodes = []
    total = 0

    while total < num_transitions:
        obs, _ = env.reset(seed=SEED + len(episodes))
        episode = defaultdict(list)
        done = False

        while not done and total < num_transitions:
            physical_action, _ = teacher.predict(obs, deterministic=True)
            normalized_action = physical_to_normalized(physical_action, env.action_space)
            next_obs, reward, terminated, truncated, _ = env.step(physical_action.astype(np.float32))
            done = terminated or truncated

            episode["states"].append(obs.astype(np.float32))
            episode["actions"].append(normalized_action.astype(np.float32))
            episode["next_states"].append(next_obs.astype(np.float32))
            episode["rewards"].append(float(reward))
            episode["dones"].append(bool(done))

            obs = next_obs
            total += 1

        episodes.append({key: np.asarray(value) for key, value in episode.items()})

    env.close()
    return episodes


def flatten_episodes(episodes):
    states = np.concatenate([episode["states"] for episode in episodes], axis=0).astype(np.float32)
    actions = np.concatenate([episode["actions"] for episode in episodes], axis=0).astype(np.float32)
    next_states = np.concatenate([episode["next_states"] for episode in episodes], axis=0).astype(np.float32)
    rewards = np.concatenate([episode["rewards"] for episode in episodes], axis=0).astype(np.float32)
    dones = np.concatenate([episode["dones"] for episode in episodes], axis=0).astype(bool)
    episode_ids = np.concatenate([
        np.full(len(episode["states"]), idx, dtype=np.int32)
        for idx, episode in enumerate(episodes)
    ])
    return states, actions, next_states, rewards, dones, episode_ids


def build_action_chunk_dataset(episodes, chunk_size=ACTION_CHUNK_SIZE):
    chunk_states = []
    chunk_actions = []
    for episode in episodes:
        states = episode["states"]
        actions = episode["actions"]
        if len(states) < chunk_size:
            continue
        for start in range(len(states) - chunk_size + 1):
            chunk_states.append(states[start])
            chunk_actions.append(actions[start:start + chunk_size])
    return np.asarray(chunk_states, dtype=np.float32), np.asarray(chunk_actions, dtype=np.float32)


def train_single_step_bc(model, states, actions, epochs=BC_EPOCHS, batch_size=256, learning_rate=1e-3):
    dataset = TensorDataset(torch.from_numpy(states), torch.from_numpy(actions))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    print("====== Single-step BC supervised training ======")
    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for batch_states, batch_actions in dataloader:
            pred_actions = model.forward_tensor(batch_states)
            loss = criterion(pred_actions, batch_actions)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch_states.size(0)
        total_loss /= len(dataset)
        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d}/{epochs} - MSE Loss: {total_loss:.6f}")
    return model.freeze()


def train_action_chunk_bc(model, states, chunks, epochs=CHUNK_BC_EPOCHS, batch_size=256, learning_rate=1e-3):
    dataset = TensorDataset(torch.from_numpy(states), torch.from_numpy(chunks))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    print("====== Action-chunk BC supervised training ======")
    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for batch_states, batch_chunks in dataloader:
            pred_chunks = model.forward_tensor(batch_states)
            loss = criterion(pred_chunks, batch_chunks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch_states.size(0)
        total_loss /= len(dataset)
        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d}/{epochs} - MSE Loss: {total_loss:.6f}")
    return model.freeze()


def evaluate_bc_controller(controller, episodes=10, name="BC"):
    env = gym.make(ENV_NAME)
    rewards = []

    for episode_idx in range(episodes):
        if hasattr(controller, "reset"):
            controller.reset()
        obs, _ = env.reset(seed=SEED + 10_000 + episode_idx)
        total_reward = 0.0
        done = False

        while not done:
            normalized_action = controller.predict(obs)
            physical_action = normalized_to_physical(normalized_action, env.action_space)
            obs, reward, terminated, truncated, _ = env.step(physical_action.astype(np.float32))
            total_reward += reward
            done = terminated or truncated

        rewards.append(total_reward)
        print(f"{name} eval episode {episode_idx + 1}: {total_reward:.2f}")

    env.close()
    print(f"{name} average reward: {np.mean(rewards):.2f} +/- {np.std(rewards):.2f}")


set_seed()

probe_env = gym.make(ENV_NAME)
state_dim = probe_env.observation_space.shape[0]
action_dim = probe_env.action_space.shape[0]
probe_env.close()

print("Training PPO teacher used only to generate expert demonstrations...")
teacher = train_teacher()

print("Collecting expert demonstrations D_expert...")
episodes = collect_expert_episodes(ENV_NAME, teacher)
states, actions, next_states, rewards, dones, episode_ids = flatten_episodes(episodes)
obs_mean = states.mean(axis=0).astype(np.float32)
obs_std = (states.std(axis=0) + 1e-6).astype(np.float32)
np.savez(
    DEMO_PATH,
    states=states,
    actions=actions,
    next_states=next_states,
    rewards=rewards,
    dones=dones,
    episode_ids=episode_ids,
    obs_mean=obs_mean,
    obs_std=obs_std,
)
print(f"Saved {len(states)} expert transitions to {DEMO_PATH}")

bc_model = BCPolicy(state_dim, action_dim)
bc_model = train_single_step_bc(bc_model, states, actions)
torch.save(bc_model.state_dict(), BC_PATH)
print(f"Saved single-step BC weights to {BC_PATH}")
evaluate_bc_controller(bc_model, name="Single-step BC")

chunk_states, chunk_targets = build_action_chunk_dataset(episodes)
chunk_bc_model = ActionChunkBC(state_dim, action_dim, chunk_size=ACTION_CHUNK_SIZE)
chunk_bc_model = train_action_chunk_bc(chunk_bc_model, chunk_states, chunk_targets)
torch.save(chunk_bc_model.state_dict(), ACTION_CHUNK_BC_PATH)
print(f"Saved action-chunk BC weights to {ACTION_CHUNK_BC_PATH}")
evaluate_bc_controller(ChunkedBCController(chunk_bc_model), name="Action-chunk BC")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 18.4 MB/s eta 0:00:00
Mounted at /content/drive


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Training PPO teacher used only to generate expert demonstrations...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved 30000 expert transitions to /content/drive/MyDrive/BC_RL_project/expert_demonstrations.npz
====== Single-step BC supervised training ======
Epoch 001/100 - MSE Loss: 0.013822
Epoch 010/100 - MSE Loss: 0.000465
Epoch 020/100 - MSE Loss: 0.000166
Epoch 030/100 - MSE Loss: 0.000128
Epoch 040/100 - MSE Loss: 0.000102
Epoch 050/100 - MSE Loss: 0.000063
Epoch 060/100 - MSE Loss: 0.000052
Epoch 070/100 - MSE Loss: 0.000044
Epoch 080/100 - MSE Loss: 0.000025
Epoch 090/100 - MSE Loss: 0.000031
Epoch 100/100 - MSE Loss: 0.000018
Saved single-step BC weights to /content/drive/MyDrive/BC_RL_project/bc_expert.pth
Single-step BC eval episode 1: -138.27
Single-step BC eval episode 2: -129.77
Single-step BC eval episode 3: -0.98
Single-step BC eval episode 4: -244.09
Single-step BC eval episode 5: -0.94
Single-step BC eval episode 6: -246.62
Single-step BC eval episode 7: -1.81
Single-step BC eval episode 8: -271.72
Single-step BC eval episode 9: -0.78
Single-step BC eval episode 10: -252.85
Sin

## Phase II Baseline: Frozen MLP BC + PPO Residual Corrector

In [ ]:
!pip install -q "stable-baselines3[extra]" gymnasium tensorboard

import os
import random
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from google.colab import drive
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv



SEED = 7
ENV_NAME = "Pendulum-v1"
BC_HIDDEN_DIMS = (256, 256)

DRIVE_DIR = "/content/drive/MyDrive/BC_RL_project"
os.makedirs(DRIVE_DIR, exist_ok=True)

BC_PATH = os.path.join(DRIVE_DIR, "bc_expert.pth")
PPO_PATH = os.path.join(DRIVE_DIR, "residual_ppo_state_dependent_final")
PPO_BEST_DIR = os.path.join(DRIVE_DIR, "best_ppo_state_dependent")
PPO_LOG_DIR = os.path.join(DRIVE_DIR, "ppo_state_dependent_eval_logs")

PPO_DISTURBANCE_BIAS = 0.20
PPO_VELOCITY_GAIN = 0.04
COMPENSATION_PRIOR = -PPO_DISTURBANCE_BIAS / 2.0

PPO_LEARNED_EPSILON = 0.08
PPO_RESIDUAL_LAMBDA = 0.001


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalized_to_physical(normalized_action, action_space):
    normalized_action = np.asarray(normalized_action, dtype=np.float32)
    action_low = action_space.low.astype(np.float32)
    action_high = action_space.high.astype(np.float32)
    midpoint = (action_high + action_low) / 2.0
    scale = (action_high - action_low) / 2.0
    return midpoint + np.clip(normalized_action, -1.0, 1.0) * scale


class BCPolicy(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims=BC_HIDDEN_DIMS):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
        )
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dims[1], action_dim),
            nn.Tanh(),
        )

    def forward_tensor(self, state_tensor):
        return self.policy_head(self.encoder(state_tensor))

    def predict(self, state):
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32)
            squeeze = state_tensor.ndim == 1
            if squeeze:
                state_tensor = state_tensor.unsqueeze(0)
            action = self.forward_tensor(state_tensor).cpu().numpy().astype(np.float32)
        return action[0] if squeeze else action

    def freeze(self):
        self.eval()
        for param in self.parameters():
            param.requires_grad = False
        return self


class StateDependentDisturbanceWrapper(gym.Wrapper):
    def __init__(self, env, action_bias=0.0, velocity_gain=0.04):
        super().__init__(env)
        self.action_bias = float(action_bias)
        self.velocity_gain = float(velocity_gain)
        self.current_obs = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.current_obs = obs.astype(np.float32)
        return self.current_obs, info

    def step(self, action):
        action = np.asarray(action, dtype=np.float32)

        theta_dot = float(self.current_obs[2])
        disturbance = self.action_bias + self.velocity_gain * theta_dot

        disturbed_action = action + disturbance
        disturbed_action = np.clip(
            disturbed_action,
            self.env.action_space.low,
            self.env.action_space.high,
        ).astype(np.float32)

        obs, reward, terminated, truncated, info = self.env.step(disturbed_action)
        self.current_obs = obs.astype(np.float32)

        info = dict(info)
        info["physical_disturbance"] = float(disturbance)
        return self.current_obs, reward, terminated, truncated, info


class ResidualControlWrapper(gym.Wrapper):
    def __init__(
        self,
        env,
        bc_policy,
        learned_epsilon=0.12,
        lamda=0.0005,
        residual_prior=0.0,
        debug=False,
    ):
        super().__init__(env)
        self.bc_policy = bc_policy
        self.learned_epsilon = float(learned_epsilon)
        self.lamda = float(lamda)
        self.residual_prior = float(residual_prior)
        self.debug = debug
        self.step_count = 0
        self.current_obs = None

        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=env.action_space.shape,
            dtype=np.float32,
        )

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.current_obs = obs.astype(np.float32)
        return self.current_obs, info

    def step(self, residual_action):
        self.step_count += 1

        base_action = np.asarray(self.bc_policy.predict(self.current_obs), dtype=np.float32)
        residual_action = np.asarray(residual_action, dtype=np.float32).reshape(self.action_space.shape)

        prior_delta = np.full(self.action_space.shape, self.residual_prior, dtype=np.float32)
        learned_delta = np.clip(residual_action, -1.0, 1.0) * self.learned_epsilon
        delta_action = np.clip(prior_delta + learned_delta, -1.0, 1.0)

        normalized_action = np.clip(base_action + delta_action, -1.0, 1.0)
        physical_action = normalized_to_physical(normalized_action, self.env.action_space)

        obs, reward, terminated, truncated, info = self.env.step(physical_action.astype(np.float32))
        self.current_obs = obs.astype(np.float32)

        residual_penalty = self.lamda * float(
            np.sum(np.square(learned_delta)) / (self.learned_epsilon ** 2 + 1e-8)
        )
        shaped_reward = float(reward) - residual_penalty

        info = dict(info)
        info.update(
            raw_reward=float(reward),
            residual_penalty=residual_penalty,
            residual_prior=prior_delta.copy(),
            learned_delta=learned_delta.copy(),
            delta_action=delta_action.copy(),
            base_action=base_action.copy(),
        )

        if self.debug and self.step_count % 1000 == 0:
            print(
                f"[Residual Debug] step={self.step_count}, "
                f"prior={float(np.mean(prior_delta)):.3f}, "
                f"learned={float(np.mean(learned_delta)):.3f}, "
                f"delta={float(np.mean(delta_action)):.3f}, "
                f"reward={float(reward):.3f}, "
                f"penalty={residual_penalty:.5f}"
            )

        return self.current_obs, shaped_reward, terminated, truncated, info


def load_single_step_bc():
    env = gym.make(ENV_NAME)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    env.close()

    model = BCPolicy(state_dim, action_dim, hidden_dims=BC_HIDDEN_DIMS)
    model.load_state_dict(torch.load(BC_PATH, map_location="cpu"))
    return model.freeze()


def make_ppo_env(
    debug=False,
    lamda=PPO_RESIDUAL_LAMBDA,
    disturbance_bias=0.0,
    velocity_gain=0.0,
    residual_prior=0.0,
):
    env = gym.make(ENV_NAME)

    if abs(disturbance_bias) > 0.0 or abs(velocity_gain) > 0.0:
        env = StateDependentDisturbanceWrapper(
            env,
            action_bias=disturbance_bias,
            velocity_gain=velocity_gain,
        )

    env = ResidualControlWrapper(
        env,
        bc_model,
        learned_epsilon=PPO_LEARNED_EPSILON,
        lamda=lamda,
        residual_prior=residual_prior,
        debug=debug,
    )

    return Monitor(env)


def evaluate_residual_model(model, env_factory, episodes=50, name="Hybrid"):
    env = env_factory()

    episode_rewards = []
    learned_mags = []
    delta_mags = []

    for episode in range(episodes):
        obs, _ = env.reset(seed=SEED + 20_000 + episode)
        total_task_reward = 0.0
        done = False

        while not done:
            if model is None:
                residual_action = np.zeros(env.action_space.shape, dtype=np.float32)
            else:
                residual_action, _ = model.predict(obs, deterministic=True)

            obs, reward, terminated, truncated, info = env.step(residual_action)
            total_task_reward += float(info.get("raw_reward", reward))

            if "learned_delta" in info:
                learned_mags.append(float(np.mean(np.abs(info["learned_delta"]))))
            if "delta_action" in info:
                delta_mags.append(float(np.mean(np.abs(info["delta_action"]))))

            done = terminated or truncated

        episode_rewards.append(total_task_reward)
        print(f"{name} eval episode {episode + 1}: {total_task_reward:.2f}")

    env.close()

    print(f"{name} average task reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
    if learned_mags:
        print(f"{name} mean |learned_delta|: {np.mean(learned_mags):.4f}")
    if delta_mags:
        print(f"{name} mean |total_delta|: {np.mean(delta_mags):.4f}")

    return episode_rewards


set_seed()

bc_model = load_single_step_bc()
print(f"Loaded frozen single-step BC weights from {BC_PATH}")
print(f"Compensation prior: {COMPENSATION_PRIOR:.4f}")
print(f"Velocity gain: {PPO_VELOCITY_GAIN:.4f}")


evaluate_residual_model(
    None,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=0.0,
        velocity_gain=0.0,
        residual_prior=0.0,
    ),
    episodes=50,
    name="Zero residual / frozen BC (clean)",
)

evaluate_residual_model(
    None,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=0.0,
    ),
    episodes=50,
    name="Zero residual / frozen BC (state-dependent disturbed)",
)

evaluate_residual_model(
    None,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=COMPENSATION_PRIOR,
    ),
    episodes=50,
    name="Prior compensation only (state-dependent disturbed)",
)


train_env = DummyVecEnv([
    lambda: make_ppo_env(
        debug=False,
        lamda=PPO_RESIDUAL_LAMBDA,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=COMPENSATION_PRIOR,
    )
])

eval_env = DummyVecEnv([
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=COMPENSATION_PRIOR,
    )
])

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=PPO_BEST_DIR,
    log_path=PPO_LOG_DIR,
    eval_freq=5_000,
    n_eval_episodes=50,
    deterministic=True,
    render=False,
)

ppo_model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=5e-5,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.1,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(
        activation_fn=nn.Tanh,
        net_arch=dict(pi=[64, 64], vf=[128, 128]),
        log_std_init=-1.8,
    ),
    verbose=1,
    seed=SEED,
    device="cpu",
)

print("Training PPO residual corrector...")
ppo_model.learn(total_timesteps=300_000, callback=eval_callback)
ppo_model.save(PPO_PATH)
print(f"Saved final PPO residual policy to {PPO_PATH}")

best_ppo = PPO.load(f"{PPO_BEST_DIR}/best_model", env=train_env, device="cpu")


evaluate_residual_model(
    best_ppo,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=0.0,
        velocity_gain=0.0,
        residual_prior=0.0,
    ),
    episodes=50,
    name="PPO residual hybrid (clean)",
)

evaluate_residual_model(
    best_ppo,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=COMPENSATION_PRIOR,
    ),
    episodes=50,
    name="PPO residual hybrid (state-dependent disturbed)",
)

Loaded frozen single-step BC weights from /content/drive/MyDrive/BC_RL_project/bc_expert.pth
Compensation prior: -0.1000
Velocity gain: 0.0400
Zero residual / frozen BC (clean) eval episode 1: -134.12
Zero residual / frozen BC (clean) eval episode 2: -389.75
Zero residual / frozen BC (clean) eval episode 3: -127.04
Zero residual / frozen BC (clean) eval episode 4: -129.19
Zero residual / frozen BC (clean) eval episode 5: -249.10
Zero residual / frozen BC (clean) eval episode 6: -256.31
Zero residual / frozen BC (clean) eval episode 7: -122.27
Zero residual / frozen BC (clean) eval episode 8: -131.18
Zero residual / frozen BC (clean) eval episode 9: -126.23
Zero residual / frozen BC (clean) eval episode 10: -127.45
Zero residual / frozen BC (clean) eval episode 11: -1.03
Zero residual / frozen BC (clean) eval episode 12: -128.03
Zero residual / frozen BC (clean) eval episode 13: -252.33
Zero residual / frozen BC (clean) eval episode 14: -128.08
Zero residual / frozen BC (clean) eval epi

[-132.85224357801158,
 -261.72384905991953,
 -126.32780149373103,
 -128.77155834372752,
 -246.78935530076666,
 -379.77812280928566,
 -121.24241727583212,
 -129.9554759681566,
 -644.8198601630055,
 -127.01857024987389,
 -0.9446416379768932,
 -126.92855043461074,
 -250.24687409794674,
 -127.02486054255361,
 -128.4343170684653,
 -0.9222926502169334,
 -121.69705854409527,
 -340.0734974974491,
 -2.732761282460727,
 -127.17726819206052,
 -1.0955274731190292,
 -259.7388998557739,
 -245.14833243435294,
 -126.96922488121082,
 -123.196047165455,
 -124.7576540497859,
 -129.11059642720105,
 -393.353925429391,
 -128.44283506625442,
 -121.47669564328099,
 -377.39911762116014,
 -136.3223861156789,
 -254.40995709421352,
 -256.5419970360131,
 -245.59177632287611,
 -128.3998348698786,
 -645.1178318367163,
 -2.1677833009118004,
 -245.28417299020316,
 -125.69978128720557,
 -126.04290333050908,
 -1.6556772911012683,
 -257.138109260361,
 -128.25336586756984,
 -240.0489639906059,
 -121.7803472897071,
 -278.9

In [ ]:
!pip install -q "stable-baselines3[extra]" gymnasium tensorboard

import os
import random

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from google.colab import drive
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv


drive.mount("/content/drive")

SEED = 7
ENV_NAME = "Pendulum-v1"
BC_HIDDEN_DIMS = (256, 256)

DRIVE_DIR = "/content/drive/MyDrive/BC_RL_project"
os.makedirs(DRIVE_DIR, exist_ok=True)

BC_PATH = os.path.join(DRIVE_DIR, "bc_expert.pth")
PPO_PATH = os.path.join(DRIVE_DIR, "residual_ppo_no_prior_final")
PPO_BEST_DIR = os.path.join(DRIVE_DIR, "best_ppo_no_prior")
PPO_LOG_DIR = os.path.join(DRIVE_DIR, "ppo_no_prior_eval_logs")

# State-dependent disturbance.
# This makes BC fail on some states and gives PPO residual something to learn.
PPO_DISTURBANCE_BIAS = 0.20
PPO_VELOCITY_GAIN = 0.04

# No prior: PPO must learn the whole residual correction.
COMPENSATION_PRIOR = 0.0

PPO_LEARNED_EPSILON = 0.20
PPO_RESIDUAL_LAMBDA = 0.0005


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalized_to_physical(normalized_action, action_space):
    normalized_action = np.asarray(normalized_action, dtype=np.float32)
    action_low = action_space.low.astype(np.float32)
    action_high = action_space.high.astype(np.float32)
    midpoint = (action_high + action_low) / 2.0
    scale = (action_high - action_low) / 2.0
    return midpoint + np.clip(normalized_action, -1.0, 1.0) * scale


class BCPolicy(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims=BC_HIDDEN_DIMS):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
        )
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dims[1], action_dim),
            nn.Tanh(),
        )

    def forward_tensor(self, state_tensor):
        return self.policy_head(self.encoder(state_tensor))

    def predict(self, state):
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32)
            squeeze = state_tensor.ndim == 1
            if squeeze:
                state_tensor = state_tensor.unsqueeze(0)
            action = self.forward_tensor(state_tensor).cpu().numpy().astype(np.float32)
        return action[0] if squeeze else action

    def freeze(self):
        self.eval()
        for param in self.parameters():
            param.requires_grad = False
        return self


class StateDependentDisturbanceWrapper(gym.Wrapper):
    def __init__(self, env, action_bias=0.0, velocity_gain=0.04):
        super().__init__(env)
        self.action_bias = float(action_bias)
        self.velocity_gain = float(velocity_gain)
        self.current_obs = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.current_obs = obs.astype(np.float32)
        return self.current_obs, info

    def step(self, action):
        action = np.asarray(action, dtype=np.float32)

        theta_dot = float(self.current_obs[2])
        disturbance = self.action_bias + self.velocity_gain * theta_dot

        disturbed_action = action + disturbance
        disturbed_action = np.clip(
            disturbed_action,
            self.env.action_space.low,
            self.env.action_space.high,
        ).astype(np.float32)

        obs, reward, terminated, truncated, info = self.env.step(disturbed_action)
        self.current_obs = obs.astype(np.float32)

        info = dict(info)
        info["physical_disturbance"] = float(disturbance)

        return self.current_obs, reward, terminated, truncated, info


class ResidualControlWrapper(gym.Wrapper):
    """
    Hybrid control:
        normalized_action = BC(s) + learned_delta

    No prior is used here, so PPO must learn the residual correction.
    """

    def __init__(
        self,
        env,
        bc_policy,
        learned_epsilon=0.20,
        lamda=0.0005,
        residual_prior=0.0,
        debug=False,
    ):
        super().__init__(env)
        self.bc_policy = bc_policy
        self.learned_epsilon = float(learned_epsilon)
        self.lamda = float(lamda)
        self.residual_prior = float(residual_prior)
        self.debug = debug
        self.step_count = 0
        self.current_obs = None

        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=env.action_space.shape,
            dtype=np.float32,
        )

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.current_obs = obs.astype(np.float32)
        return self.current_obs, info

    def step(self, residual_action):
        self.step_count += 1

        base_action = np.asarray(self.bc_policy.predict(self.current_obs), dtype=np.float32)
        residual_action = np.asarray(residual_action, dtype=np.float32).reshape(self.action_space.shape)

        prior_delta = np.full(self.action_space.shape, self.residual_prior, dtype=np.float32)
        learned_delta = np.clip(residual_action, -1.0, 1.0) * self.learned_epsilon
        delta_action = np.clip(prior_delta + learned_delta, -1.0, 1.0)

        normalized_action = np.clip(base_action + delta_action, -1.0, 1.0)
        physical_action = normalized_to_physical(normalized_action, self.env.action_space)

        obs, reward, terminated, truncated, info = self.env.step(physical_action.astype(np.float32))
        self.current_obs = obs.astype(np.float32)

        residual_penalty = self.lamda * float(
            np.sum(np.square(learned_delta)) / (self.learned_epsilon ** 2 + 1e-8)
        )

        shaped_reward = float(reward) - residual_penalty

        info = dict(info)
        info.update(
            raw_reward=float(reward),
            residual_penalty=residual_penalty,
            residual_prior=prior_delta.copy(),
            learned_delta=learned_delta.copy(),
            delta_action=delta_action.copy(),
            base_action=base_action.copy(),
        )

        if self.debug and self.step_count % 1000 == 0:
            print(
                f"[Residual Debug] step={self.step_count}, "
                f"learned={float(np.mean(learned_delta)):.3f}, "
                f"delta={float(np.mean(delta_action)):.3f}, "
                f"reward={float(reward):.3f}, "
                f"penalty={residual_penalty:.5f}"
            )

        return self.current_obs, shaped_reward, terminated, truncated, info


def load_single_step_bc():
    env = gym.make(ENV_NAME)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    env.close()

    model = BCPolicy(state_dim, action_dim, hidden_dims=BC_HIDDEN_DIMS)
    model.load_state_dict(torch.load(BC_PATH, map_location="cpu"))
    return model.freeze()


def make_ppo_env(
    debug=False,
    lamda=PPO_RESIDUAL_LAMBDA,
    disturbance_bias=0.0,
    velocity_gain=0.0,
    residual_prior=COMPENSATION_PRIOR,
):
    env = gym.make(ENV_NAME)

    if abs(disturbance_bias) > 0.0 or abs(velocity_gain) > 0.0:
        env = StateDependentDisturbanceWrapper(
            env,
            action_bias=disturbance_bias,
            velocity_gain=velocity_gain,
        )

    env = ResidualControlWrapper(
        env,
        bc_model,
        learned_epsilon=PPO_LEARNED_EPSILON,
        lamda=lamda,
        residual_prior=residual_prior,
        debug=debug,
    )

    return Monitor(env)


def evaluate_residual_model(model, env_factory, episodes=50, name="Hybrid"):
    env = env_factory()

    episode_rewards = []
    learned_mags = []
    delta_mags = []

    for episode in range(episodes):
        obs, _ = env.reset(seed=SEED + 20_000 + episode)
        total_task_reward = 0.0
        done = False

        while not done:
            if model is None:
                residual_action = np.zeros(env.action_space.shape, dtype=np.float32)
            else:
                residual_action, _ = model.predict(obs, deterministic=True)

            obs, reward, terminated, truncated, info = env.step(residual_action)
            total_task_reward += float(info.get("raw_reward", reward))

            if "learned_delta" in info:
                learned_mags.append(float(np.mean(np.abs(info["learned_delta"]))))
            if "delta_action" in info:
                delta_mags.append(float(np.mean(np.abs(info["delta_action"]))))

            done = terminated or truncated

        episode_rewards.append(total_task_reward)
        print(f"{name} eval episode {episode + 1}: {total_task_reward:.2f}")

    env.close()

    print(f"{name} average task reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
    if learned_mags:
        print(f"{name} mean |learned_delta|: {np.mean(learned_mags):.4f}")
    if delta_mags:
        print(f"{name} mean |total_delta|: {np.mean(delta_mags):.4f}")

    return episode_rewards


set_seed()

bc_model = load_single_step_bc()
print(f"Loaded frozen single-step BC weights from {BC_PATH}")
print(f"Disturbance bias: {PPO_DISTURBANCE_BIAS:.4f}")
print(f"Velocity gain: {PPO_VELOCITY_GAIN:.4f}")
print(f"No prior compensation: {COMPENSATION_PRIOR:.4f}")


evaluate_residual_model(
    None,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=0.0,
        velocity_gain=0.0,
        residual_prior=0.0,
    ),
    episodes=50,
    name="Zero residual / frozen BC (clean)",
)

evaluate_residual_model(
    None,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=0.0,
    ),
    episodes=50,
    name="Zero residual / frozen BC (state-dependent disturbed)",
)


train_env = DummyVecEnv([
    lambda: make_ppo_env(
        debug=False,
        lamda=PPO_RESIDUAL_LAMBDA,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=0.0,
    )
])

eval_env = DummyVecEnv([
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=0.0,
    )
])

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=PPO_BEST_DIR,
    log_path=PPO_LOG_DIR,
    eval_freq=5_000,
    n_eval_episodes=50,
    deterministic=True,
    render=False,
)

ppo_model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=1e-4,
    n_steps=1024,
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.1,
    ent_coef=0.0,
    policy_kwargs=dict(
        activation_fn=nn.Tanh,
        net_arch=dict(pi=[128, 128], vf=[128, 128]),
        log_std_init=-1.5,
    ),
    verbose=0,
    seed=SEED,
    device="cpu",
)

print("Training PPO residual corrector without prior...")
ppo_model.learn(total_timesteps=500_000, callback=eval_callback)
ppo_model.save(PPO_PATH)
print(f"Saved final PPO residual policy to {PPO_PATH}")

best_ppo = PPO.load(f"{PPO_BEST_DIR}/best_model", env=train_env, device="cpu")


evaluate_residual_model(
    best_ppo,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=0.0,
        velocity_gain=0.0,
        residual_prior=0.0,
    ),
    episodes=50,
    name="PPO residual hybrid (clean)",
)

evaluate_residual_model(
    best_ppo,
    lambda: make_ppo_env(
        debug=False,
        lamda=0.0,
        disturbance_bias=PPO_DISTURBANCE_BIAS,
        velocity_gain=PPO_VELOCITY_GAIN,
        residual_prior=0.0,
    ),
    episodes=50,
    name="PPO residual hybrid (state-dependent disturbed)",
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded frozen single-step BC weights from /content/drive/MyDrive/BC_RL_project/bc_expert.pth
Disturbance bias: 0.2000
Velocity gain: 0.0400
No prior compensation: 0.0000
Zero residual / frozen BC (clean) eval episode 1: -134.12
Zero residual / frozen BC (clean) eval episode 2: -389.75
Zero residual / frozen BC (clean) eval episode 3: -127.04
Zero residual / frozen BC (clean) eval episode 4: -129.19
Zero residual / frozen BC (clean) eval episode 5: -249.10
Zero residual / frozen BC (clean) eval episode 6: -256.31
Zero residual / frozen BC (clean) eval episode 7: -122.27
Zero residual / frozen BC (clean) eval episode 8: -131.18
Zero residual / frozen BC (clean) eval episode 9: -126.23
Zero residual / frozen BC (clean) eval episode 10: -127.45
Zero residual / frozen BC (clean) eval episode 11: -1.03
Zero residual / frozen BC (clean) eval episode 12: -128.03
Zero

[-134.67872239170526,
 -1046.2642761617064,
 -129.86896533642584,
 -1006.8595220412284,
 -250.50548375907422,
 -383.15981065312496,
 -123.25602498238035,
 -131.72482453801052,
 -1055.3880301635543,
 -130.22838020681405,
 -4.193987818472123,
 -130.90942478480045,
 -254.69294898936607,
 -130.95941808338736,
 -1043.1806914099807,
 -4.198213500067412,
 -123.70530011868934,
 -361.6636569617042,
 -5.273577414280068,
 -131.10818269320367,
 -4.279348211240988,
 -1040.298326310311,
 -980.3895517187634,
 -131.09136439855274,
 -125.46467384269947,
 -128.9168560782697,
 -1049.5597393150829,
 -399.4939461785126,
 -1048.2379359986483,
 -125.10288944761898,
 -380.36097461872885,
 -137.80858948123245,
 -257.20961884112637,
 -259.5410345029235,
 -249.0590777666062,
 -130.74204114708192,
 -1044.6821696259697,
 -5.674410538425298,
 -249.86844915754102,
 -1040.359095397519,
 -129.69322135616227,
 -4.863285339963365,
 -262.2455450973956,
 -132.01344041736013,
 -243.73935489212462,
 -123.16776467483486,
 -2

## Advanced Paradigm: Frozen Action-Chunk BC + SAC Residual Corrector

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv


SAC_PATH = "residual_sac_corrector"
SAC_BEST_DIR = "./best_sac_residual"
SAC_LOG_DIR = "./sac_eval_logs"
SAC_EPSILON = 0.15
SAC_LAMBDA = 0.005
SAC_DISTURBANCE_BIAS = 0.25
SAC_DISTURBANCE_NOISE = 0.02


def load_action_chunk_bc():
    env = gym.make(ENV_NAME)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    env.close()

    model = ActionChunkBC(
        state_dim,
        action_dim,
        chunk_size=ACTION_CHUNK_SIZE,
        hidden_dims=BC_HIDDEN_DIMS,
    )
    model.load_state_dict(torch.load(ACTION_CHUNK_BC_PATH, map_location="cpu"))
    return model.freeze()


demo_data = np.load(DEMO_PATH)
obs_mean = demo_data["obs_mean"]
obs_std = demo_data["obs_std"]
chunk_bc_model = load_action_chunk_bc()
print(f"Loaded frozen action-chunk BC weights from {ACTION_CHUNK_BC_PATH}")


def make_sac_env(debug=False, lamda=SAC_LAMBDA, disturbance_bias=0.0, disturbance_noise=0.0):
    env = gym.make(ENV_NAME)
    if abs(disturbance_bias) > 0.0 or disturbance_noise > 0.0:
        env = ActionDisturbanceWrapper(
            env,
            action_bias=disturbance_bias,
            action_noise_std=disturbance_noise,
        )
    env = ResidualControlWrapper(
        env,
        ChunkedBCController(chunk_bc_model),
        epsilon=SAC_EPSILON,
        lamda=lamda,
        adaptive_bound=True,
        obs_mean=obs_mean,
        obs_std=obs_std,
        epsilon_max=0.40,
        ood_threshold=2.0,
        uncertainty_gain=0.35,
        debug=debug,
    )
    return Monitor(env)


def residual_targets_from_expert_data(
    data,
    chunk_policy,
    epsilon=SAC_EPSILON,
    lamda=SAC_LAMBDA,
    disturbance_bias=SAC_DISTURBANCE_BIAS,
):
    states = data["states"].astype(np.float32)
    expert_actions = data["actions"].astype(np.float32)
    next_states = data["next_states"].astype(np.float32)
    raw_rewards = data["rewards"].astype(np.float32)
    dones = data["dones"].astype(bool)
    episode_ids = data["episode_ids"].astype(np.int32)

    controller = ChunkedBCController(chunk_policy)
    # Pendulum action space is symmetric [-2, 2], so physical bias / 2 maps
    # into the normalized action coordinates used by the BC and residual policy.
    disturbance_bias_norm = np.asarray([disturbance_bias / 2.0], dtype=np.float32)
    residual_actions = []
    shaped_rewards = []
    last_episode = None

    for idx, obs in enumerate(states):
        if last_episode is None or episode_ids[idx] != last_episode:
            controller.reset()
            last_episode = episode_ids[idx]

        base_action = controller.predict(obs)
        delta_action = np.clip(expert_actions[idx] - base_action - disturbance_bias_norm, -epsilon, epsilon)
        residual_action = np.clip(delta_action / (epsilon + 1e-8), -1.0, 1.0).astype(np.float32)
        penalty = lamda * float(np.sum(np.square(delta_action)) / (epsilon ** 2 + 1e-8))
        residual_actions.append(residual_action)
        shaped_rewards.append(float(raw_rewards[idx]) - penalty)

    return (
        states,
        next_states,
        np.asarray(residual_actions, dtype=np.float32),
        np.asarray(shaped_rewards, dtype=np.float32),
        dones,
    )


def bootstrap_sac_replay_buffer(model, data, chunk_policy):
    states, next_states, residual_actions, shaped_rewards, dones = residual_targets_from_expert_data(
        data,
        chunk_policy,
        disturbance_bias=SAC_DISTURBANCE_BIAS,
    )
    for idx in range(len(states)):
        model.replay_buffer.add(
            states[idx][None, :],
            next_states[idx][None, :],
            residual_actions[idx][None, :],
            np.asarray([shaped_rewards[idx]], dtype=np.float32),
            np.asarray([dones[idx]], dtype=bool),
            infos=[{}],
        )
    print(f"Replay buffer bootstrapped with {len(states)} expert residual transitions.")


evaluate_residual_model(
    None,
    lambda: make_sac_env(debug=False, lamda=0.0, disturbance_bias=0.0, disturbance_noise=0.0),
    name="Zero residual / action-chunk BC (clean)",
)
evaluate_residual_model(
    None,
    lambda: make_sac_env(debug=False, lamda=0.0, disturbance_bias=SAC_DISTURBANCE_BIAS, disturbance_noise=SAC_DISTURBANCE_NOISE),
    name="Zero residual / action-chunk BC (disturbed)",
)

train_env = DummyVecEnv([
    lambda: make_sac_env(debug=False, lamda=SAC_LAMBDA, disturbance_bias=SAC_DISTURBANCE_BIAS, disturbance_noise=SAC_DISTURBANCE_NOISE)
])
eval_env = DummyVecEnv([
    lambda: make_sac_env(debug=False, lamda=0.0, disturbance_bias=SAC_DISTURBANCE_BIAS, disturbance_noise=SAC_DISTURBANCE_NOISE)
])

sac_model = SAC(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    buffer_size=100_000,
    learning_starts=0,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    ent_coef="auto",
    train_freq=1,
    gradient_steps=1,
    policy_kwargs=dict(
        activation_fn=nn.ReLU,
        net_arch=[256, 256],
    ),
    verbose=0,
    seed=SEED,
    device="cpu",
)

print("Bootstrapping SAC replay buffer from D_expert residual targets...")
bootstrap_sac_replay_buffer(sac_model, demo_data, chunk_bc_model)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=SAC_BEST_DIR,
    log_path=SAC_LOG_DIR,
    eval_freq=10_000,
    n_eval_episodes=20,
    deterministic=True,
    render=False,
)

print("Training SAC residual corrector with action-chunk BC prior...")
sac_model.learn(total_timesteps=500_000, callback=eval_callback)
sac_model.save(SAC_PATH)
print(f"Saved final SAC residual policy to {SAC_PATH}")

best_sac = SAC.load(f"{SAC_BEST_DIR}/best_model", env=train_env, device="cpu")
evaluate_residual_model(
    best_sac,
    lambda: make_sac_env(debug=False, lamda=0.0, disturbance_bias=0.0, disturbance_noise=0.0),
    name="SAC residual hybrid (clean)",
)
evaluate_residual_model(
    best_sac,
    lambda: make_sac_env(debug=False, lamda=0.0, disturbance_bias=SAC_DISTURBANCE_BIAS, disturbance_noise=SAC_DISTURBANCE_NOISE),
    name="SAC residual hybrid (disturbed)",
)

Loaded frozen action-chunk BC weights from bc_action_chunk.pth
Zero residual / action-chunk BC (clean) eval episode 1: -266.56
Zero residual / action-chunk BC (clean) eval episode 2: -253.26
Zero residual / action-chunk BC (clean) eval episode 3: -126.40
Zero residual / action-chunk BC (clean) eval episode 4: -128.61
Zero residual / action-chunk BC (clean) eval episode 5: -251.29
Zero residual / action-chunk BC (clean) eval episode 6: -391.97
Zero residual / action-chunk BC (clean) eval episode 7: -123.67
Zero residual / action-chunk BC (clean) eval episode 8: -259.84
Zero residual / action-chunk BC (clean) eval episode 9: -126.02
Zero residual / action-chunk BC (clean) eval episode 10: -128.10
Zero residual / action-chunk BC (clean) eval episode 11: -0.99
Zero residual / action-chunk BC (clean) eval episode 12: -127.73
Zero residual / action-chunk BC (clean) eval episode 13: -254.41
Zero residual / action-chunk BC (clean) eval episode 14: -127.50
Zero residual / action-chunk BC (clean